# Fine-Tuning **SmolLM-135M** model for Generating Sports News:
 
 - Utilized **AG News Dataset** for a generative task using the **Sports** news available in it.
    

In [1]:
# Importing Libraries:

import numpy as np
import pandas as pd
import torch

import datasets
from datasets import load_dataset

import transformers

# Preprocessing:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling

# Training:
from transformers import TrainingArguments, Trainer

# Post Training Analysis:
from transformers import pipeline
import evaluate
import re

/home/ashish-ml-prep/Music/Personal_Videos_(Phone)/Projects/Self_Projects/ft_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading **AG News** Dataset:


In [2]:
news_dataset = load_dataset("ag_news")
news_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
# Training Dataset and features:
news_train_dataset = news_dataset["train"]
print(news_train_dataset.features)


{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'], id=None)}


In [4]:
# Sports News:
news_train_dataset[1300]

{'text': "Phelps's chase of Spitz mark? It's history This was the event Michael Phelps didn't really need to compete in if his goal was to win eight golds. He probably would have had a better chance somewhere else.",
 'label': 1}

In [5]:
# Defining news id to label map:
news_id_to_label_map = { 0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech" }

In [6]:
# Filtering the sports news dataset using label_id:
sports_datasets = news_dataset.filter(lambda example: example["label"] == 1)
sports_datasets = sports_datasets.remove_columns("label")

## Preprocessing:

### Loading the tokenizer for SmolLM-135M:


In [7]:
# Loading tokenizer for SmolLM-135M: 
model_name = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [8]:
# We need to specify as SmolLM's tokenizer doesn't include the padding token:
tokenizer.pad_token = ( tokenizer.eos_token )  

In [9]:
# Define Tokenizer wrapper function:
def tokenizer_wrapper(batch):
    return tokenizer(batch["text"], truncation=True)


### Tokenizing the Sports News Dataset:

In [10]:
# Note: We require input_ids and attention_mask
# Since we can straight up work with token ids:

tokenized_sports_news_datasets = sports_datasets.map(
                                    tokenizer_wrapper,  
                                    batched = True, 
                                    remove_columns = ["text"],  
                                )


In [11]:
tokenized_sports_news_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1900
    })
})

In [12]:
# Showing example tokenization:
ip_index = 69
example_tokenized = tokenized_sports_news_datasets['train'][ip_index]
example_tokenized_input_ids = list(example_tokenized['input_ids'])
example_tokenized_attention_mask = list(example_tokenized['attention_mask'])

# Showing example tokenization:
print(f"tokenized_ids:\n{example_tokenized_input_ids}")
print(f"\n\nattention_mask:\n{example_tokenized_attention_mask}")



tokenized_ids:
[44745, 917, 8992, 18968, 370, 216, 35, 29, 33, 288, 48750, 31732, 534, 365, 3872, 25, 6594, 731, 41710, 8581, 12713, 3917, 582, 1658, 281, 2976, 7954, 616, 327, 650, 808, 4726, 281, 3920, 253, 3531, 284, 4573, 4653, 10463, 2994, 253, 1296, 29, 10521, 24190, 282, 260, 11554, 18968, 370, 351, 253, 216, 35, 29, 33, 9970, 10528, 30]


attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Training the Model for Fine-Tuning:


In [13]:
# Identifying device to train on GPU:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
# The parameter `mlm` ==> masked language modeling
# Since we are doing Causal Learning, we set:
# mlm = False

# Initialising the data collator:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [15]:
# Reviewing shape of the tokenized texts inputs:
samples = [tokenized_sports_news_datasets["train"][i] for i in range(5)]

for sample in samples:
    print(f"input_ids shape: {len(sample['input_ids'])}")

input_ids shape: 103
input_ids shape: 81
input_ids shape: 60
input_ids shape: 65
input_ids shape: 51


In [16]:
# Reviewing shape of the tokenized samples post using data collator:
data_collator_samples_output = data_collator(samples)
for key in data_collator_samples_output:
    print(f"{key} shape: {data_collator_samples_output[key].shape}")

input_ids shape: torch.Size([5, 103])
attention_mask shape: torch.Size([5, 103])
labels shape: torch.Size([5, 103])


### Loading the SmolLM-135M model:

In [17]:
# Loading the model (SmolLM-135M) for causal learning:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

### Setting up Training Arguments and Initialising the Trainer:

In [18]:
# Setting the Training Arguments:
batch_size = 2

training_args = TrainingArguments(
    "sports-news-generator",
    push_to_hub = False,
    per_device_train_batch_size = batch_size,
    weight_decay = 0.1,
    lr_scheduler_type = "cosine",
    learning_rate = 5e-4,
    num_train_epochs = 2,
    eval_strategy = "steps",
    eval_steps = 200,
    logging_steps = 200,
)

In [19]:
# Shuffling dataset to pick 20000 examples to Train/Fine-Tune over:
shuffled_dataset = tokenized_sports_news_datasets["train"].shuffle(seed = 69)
training_subset_data = shuffled_dataset.select(range(20000))


In [20]:
# Initialize the Trainer:
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    data_collator = data_collator,
    train_dataset = training_subset_data,
    eval_dataset = tokenized_sports_news_datasets["test"].select(range(1600)),
)

/tmp/ipykernel_570507/846669114.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start Training/Fine-Tuning the model:

In [21]:
# Train the model:
trainer.train()

Step,Training Loss,Validation Loss
200,4.247100,4.227316
400,4.094400,4.170657
600,3.994500,4.128116
800,4.008400,4.051703
1000,3.985500,4.027221
1200,3.922800,3.983801
1400,3.928800,3.951313
1600,3.942000,3.918857
1800,3.846600,3.869408
2000,3.876000,3.847630


TrainOutput(global_step=20000, training_loss=2.889854670715332, metrics={'train_runtime': 4015.6466, 'train_samples_per_second': 9.961, 'train_steps_per_second': 4.981, 'total_flos': 1645061796472320.0, 'train_loss': 2.889854670715332, 'epoch': 2.0})

### Saving the Fine-Tuned model:

In [22]:
# Saving the Fine-Tuned model in './Sports_News_Generation_model' directory:
trainer.save_model("./Sports_News_Generation_model/sports_news_gen_model_bs2_ep3_FT_latest")


## Post Training/Fine-Tuning Analysis: 

In [21]:
# Initialising pipeline for inferences:
pipe = pipeline(
    "text-generation",
    model="./Sports_News_Generation_model/sports_news_gen_model_bs2_ep3_FT_latest", 
    device=device,
)



# Test generation:
print(
    pipe("1st Quarter", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
        "generated_text"
    ]
)

Device set to use cuda


1st Quarterback, 2004, Wins MVP Battle (AP) AP - Tom Brady threw for 163 yards and two touchdowns


In [22]:
# Example generation:
input_prompt_example_1 = "Manchester City Champions League Final"
generated_example_1 = pipe(input_prompt_example_1, do_sample=True, temperature=0.1, max_new_tokens=30)[0][
                        "generated_text"
                    ]

input_prompt_example_2 = "Sacramento Kings"
generated_example_2 = pipe(input_prompt_example_2, do_sample=True, temperature=0.1, max_new_tokens=30)[0][
                        "generated_text"
                    ]


print(f"input_prompt_example_1:\n{input_prompt_example_1}\n\ngenerated_example_1:\n{generated_example_1} ")
print(f"input_prompt_example_2:\n{input_prompt_example_2}\n\ngenerated_example_2:\n{generated_example_2} ")

input_prompt_example_1:
Manchester City Champions League Final

generated_example_1:
Manchester City Champions League Finale a Successive Game The final round of the Champions League has been a success for the English soccer club, with its first defeat of the season in 
input_prompt_example_2:
Sacramento Kings

generated_example_2:
Sacramento Kings Team Report - November 16 (Sports Network) - The Sacramento Kings will try to avoid a trip to the preseason with a trip to the 


In [23]:
# Dataset for post training analysis:
test_dataset = shuffled_dataset.select(range(20000, 25000))
test_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5000
})

In [24]:
# Function to decode input_ids to text
def decode_input_ids(input_ids):
    return tokenizer.decode(input_ids, skip_special_tokens=True)

# Function to split the text into:
# - prompt (first line of the text) and
# - the rest as reference

def get_first_sentence(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    if len(sentences[0].split()) > 10:
        first_15_words = ' '.join(sentences[0].split()[:10])
        return first_15_words, text
    else:
        return sentences[0], text



In [25]:
# Using 300 samples from the test_dataset for post training analysis:
test_subset = test_dataset.select(range(300))


In [26]:
# Generating prompt-reference dataset: 
prompt_reference_data = []

for elem in test_subset:
    text = decode_input_ids(elem["input_ids"])
    prompt, reference = get_first_sentence(text)
    prompt_reference_data.append({"prompt": prompt, "reference": reference})


In [27]:
# Testing 5 samples:
for i in range(5):
    print(f"Sample {i + 1}:")
    print(f"Prompt: {prompt_reference_data[i]['prompt']}")
    print(f"Reference: {prompt_reference_data[i]['reference']}\n")


Sample 1:
Prompt: Police ID Officer in Red Sox Fan Death (AP) AP
Reference: Police ID Officer in Red Sox Fan Death (AP) AP - The officer who fired a pepper-spray pellet that killed a woman in a raucous crowd of Red Sox fans was aiming at another fan but missed, police said Friday.

Sample 2:
Prompt: Roddick Continues to Show His Dominance t has been an
Reference: Roddick Continues to Show His Dominance t has been an uninspiring United States Open for the American men, who fell apart with remarkable alacrity in the first week of the tournament.

Sample 3:
Prompt: MLB Deal Favorable To Orioles #39; Angelos Bud Selig announces
Reference: MLB Deal Favorable To Orioles #39; Angelos  Bud Selig announces that the troubled Montreal Expos will move to Washington, returning baseball to the nation #39;s capital for the 2005 season.

Sample 4:
Prompt: Let the games begin We are down to seven unbeaten
Reference: Let the games begin We are down to seven unbeaten teams with Bowl Championship Series t

In [28]:
# Declaring pipe for text generation:
pipe = pipeline("text-generation", model=model_name, device=device)


Device set to use cuda


In [29]:
# Prompt Texts:
prompts_texts = [data["prompt"] for data in prompt_reference_data]


In [30]:
# Generating completions:
# generated_texts = [pipe(data["prompt"], do_sample=True, temperature=0.7, max_new_tokens=42)[0]["generated_text"] for data in prompt_reference_data]
generated_texts_list = pipe(prompts_texts, do_sample=True, temperature=0.7, max_new_tokens=42)


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for o

In [31]:
generated_texts_list

[[{'generated_text': 'Police ID Officer in Red Sox Fan Death (AP) AP\n- Police ID Officer in Red Sox Fan Death (AP) AP\n- Police ID Officer in Red Sox Fan Death (AP) AP\n- Police ID Officer in Red Sox Fan Death ('}],
 [{'generated_text': 'Roddick Continues to Show His Dominance t has been an important influence on the development of the American Revolution.\n17th century British statesman, who is best known as the father of the American Revolution, Charles John Fox – 1775 - 1'}],
 [{'generated_text': 'MLB Deal Favorable To Orioles #39; Angelos Bud Selig announces the sale of his first house, a 200-year-old Italian chestnut farmhouse and 12 acres of farmland, near the town of Siena, Italy. The deal is a'}],
 [{'generated_text': 'Let the games begin We are down to seven unbeaten players. And the whole team does not have any of the 100-plus players as a result.\nThe team has a total of 200 players. This means that the team has'}],
 [{'generated_text': "Marino Doesn't See Return to Dolphin

In [32]:
generated_texts = [generated_text[0]["generated_text"] for generated_text in generated_texts_list]


### Calculating ROUGE and BLEU Scores:

In [33]:
# Loading BLEU and ROUGE scores from evaluate:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [34]:
# Calculating BLEU scores and ROUGE scores for all 300 samples:
bleu_scores = [bleu.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]
rouge_scores = [rouge.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]


In [35]:
# Printing BLEU and ROUGE scores for first 5 samples:
print(f"BLEU Scores for the first 5 samples: {bleu_scores[:5]}\n")
print(f"ROUGE Scores for the first 5 samples: {rouge_scores[:5]}")


BLEU Scores for the first 5 samples: [{'bleu': 0.2645675921773855, 'precisions': [0.3333333333333333, 0.2765957446808511, 0.2391304347826087, 0.2222222222222222], 'brevity_penalty': 1.0, 'length_ratio': 1.0909090909090908, 'translation_length': 48, 'reference_length': 44}, {'bleu': 0.24684360178824366, 'precisions': [0.4090909090909091, 0.27906976744186046, 0.19047619047619047, 0.17073170731707318], 'brevity_penalty': 1.0, 'length_ratio': 1.3333333333333333, 'translation_length': 44, 'reference_length': 33}, {'bleu': 0.26405854414176494, 'precisions': [0.36363636363636365, 0.2558139534883721, 0.23809523809523808, 0.21951219512195122], 'brevity_penalty': 1.0, 'length_ratio': 1.1891891891891893, 'translation_length': 44, 'reference_length': 37}, {'bleu': 0.214429589812531, 'precisions': [0.3333333333333333, 0.20454545454545456, 0.18604651162790697, 0.16666666666666666], 'brevity_penalty': 1.0, 'length_ratio': 1.2857142857142858, 'translation_length': 45, 'reference_length': 35}, {'bleu':

### For Overall BLEU and ROUGE scores:

In [36]:
prompts_texts_bleu = [[elem['prompt']] for elem in prompt_reference_data]
prompts_texts_bleu

[['Police ID Officer in Red Sox Fan Death (AP) AP'],
 ['Roddick Continues to Show His Dominance t has been an'],
 ['MLB Deal Favorable To Orioles #39; Angelos Bud Selig announces'],
 ['Let the games begin We are down to seven unbeaten'],
 ["Marino Doesn't See Return to Dolphins Soon (AP) AP -"],
 ['Carter to miss two preseason games to fight lawsuit Vince'],
 ['Death by a thousand cuts THE ghosts of Calcutta 2001'],
 ['Sorenstam returns from layoff to win 53rd LPGA Tour title'],
 ["FA Won't Punish Beckham Over Yellow Card (AP) AP -"],
 ['CAMACHO OFFERS RESIGNATION - REPORTS Real Madrid coach Jose Antonio'],
 ['Singh Keeps Lead Vijay Singh follows up his opening-round 64'],
 ['PREVIEW- #39;Real #39; tournament begins at the Oval The Champions'],
 ['Even in defeat, Seattle gains confidence But to play as'],
 ['EDITORIAL:Ichiro nears record With his fourth five-hit game this season,'],
 ['Delgado bears no grudges, but doesn #39;t think Jays made'],
 ['D-Backs increase offer to Sexson The 

In [37]:
# Get reference texts for the generated output:
reference_texts = [[elem['reference']] for elem in prompt_reference_data]
reference_texts[:5]


[['Police ID Officer in Red Sox Fan Death (AP) AP - The officer who fired a pepper-spray pellet that killed a woman in a raucous crowd of Red Sox fans was aiming at another fan but missed, police said Friday.'],
 ['Roddick Continues to Show His Dominance t has been an uninspiring United States Open for the American men, who fell apart with remarkable alacrity in the first week of the tournament.'],
 ['MLB Deal Favorable To Orioles #39; Angelos  Bud Selig announces that the troubled Montreal Expos will move to Washington, returning baseball to the nation #39;s capital for the 2005 season.'],
 ['Let the games begin We are down to seven unbeaten teams with Bowl Championship Series title game aspirations. Idealistically, all but Boise State have a chance at getting to the Orange Bowl.'],
 ["Marino Doesn't See Return to Dolphins Soon (AP) AP - Dan Marino misses being part of the Miami Dolphins, yet does not see a scenario where he'd soon consider returning to the team's front office."]]

In [38]:
generated_texts[:5]

['Police ID Officer in Red Sox Fan Death (AP) AP\n- Police ID Officer in Red Sox Fan Death (AP) AP\n- Police ID Officer in Red Sox Fan Death (AP) AP\n- Police ID Officer in Red Sox Fan Death (',
 'Roddick Continues to Show His Dominance t has been an important influence on the development of the American Revolution.\n17th century British statesman, who is best known as the father of the American Revolution, Charles John Fox – 1775 - 1',
 'MLB Deal Favorable To Orioles #39; Angelos Bud Selig announces the sale of his first house, a 200-year-old Italian chestnut farmhouse and 12 acres of farmland, near the town of Siena, Italy. The deal is a',
 'Let the games begin We are down to seven unbeaten players. And the whole team does not have any of the 100-plus players as a result.\nThe team has a total of 200 players. This means that the team has',
 "Marino Doesn't See Return to Dolphins Soon (AP) AP - March 29, 2013\n- 'Tiger' Tarantula killed in Hawaii by man, zoo officials say\n- Tarantula

In [39]:
# Overall BLEU Score:
overall_bleu_score = bleu.compute(predictions=generated_texts, references=reference_texts)
print(f"Overall BLEU Score: {overall_bleu_score['bleu']:.4f}")


Overall BLEU Score: 0.2456


In [40]:
# ROUGE scores:
rouge_1_f1 = [score['rouge1'] for score in rouge_scores]
rouge_2_f1 = [score['rouge2'] for score in rouge_scores]
rouge_l_f1 = [score['rougeL'] for score in rouge_scores]

# Calculate average ROUGE F1 scores:
average_rouge_1_f1 = np.mean(rouge_1_f1)
average_rouge_2_f1 = np.mean(rouge_2_f1)
average_rouge_l_f1 = np.mean(rouge_l_f1)

# Print the average ROUGE scores:
print(f"Average ROUGE-1 F1: {average_rouge_1_f1}")
print(f"Average ROUGE-2 F1: {average_rouge_2_f1}")
print(f"Average ROUGE-L F1: {average_rouge_l_f1}")

Average ROUGE-1 F1: 0.35428286231911427
Average ROUGE-2 F1: 0.24171362692612183
Average ROUGE-L F1: 0.3322412083189667


In [ ]:
import torch
torch.cuda.empty_cache()
